# Chat Templates

Chat templates are essential for structuring interactions between language models and users. Whether you’re building a simple chatbot or a complex AI agent, understanding how to properly format your conversations is crucial for getting the best results from your model. In this guide, we’ll explore what chat templates are, why they matter, and how to use them effectively.



## 1. Model Types and Templates


A base model is trained on raw text data to predict the next token, while an instruct model is fine-tuned specifically to follow instructions and engage in conversations. For example, SmolLM2-135M is a base model, while SmolLM2-135M-Instruct is its instruction-tuned variant.



Instruction tuned models are trained to follow a specific conversational structure, making them more suitable for chatbot applications. Moreover, instruct models can handle complex interactions, including tool use, multimodal inputs, and function calling.



To make a base model behave like an instruct model, we need to format our prompts in a consistent way that the model can understand. This is where chat templates come in. ChatML is one such template format that structures conversations with clear role indicators (system, user, assistant).

When using an instruct model, always verify you’re using the correct chat template format. Using the wrong template can result in poor model performance or unexpected behavior. The easiest way to ensure this is to check the model tokenizer configuration on the Hub.

### 1.1. Common template formats

Before diving into specific implementations, it’s important to understand how different models expect their conversations to be formatted. Let’s explore some common template formats using a simple example conversation:



We’ll use the following conversation structure for all examples:



In [2]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi! How can I help you today?"},
    {"role": "user", "content": "What's the weather?"},
]

This is the ChatML template used in models like SmolLM2 and Qwen 2:



<pre>
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hi! How can I help you today?<|im_end|>
<|im_start|>user
What's the weather?<|im_start|>assistant
</pre>

This is using the `mistral` template format:



<pre>
&lt;s&gt;[INST] You are a helpful assistant. [/INST]
Hi! How can I help you today?&lt;/s&gt;
[INST] Hello! [/INST]
</pre>

Understanding these differences is key to working with various models. Let’s look at how the transformers library helps us handle these variations automatically:



In [5]:
!pip install tiktoken

   ---------------------------------------- 0.0/879.4 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/879.4 kB ? eta -:--:--
   ---------------------------------------- 879.4/879.4 kB 4.0 MB/s  0:00:00


In [7]:
from transformers import AutoTokenizer

mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True)
smol_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
]

# Each will format according to its model's template
mistral_chat = mistral_tokenizer.apply_chat_template(messages, tokenize=False)
smol_chat = smol_tokenizer.apply_chat_template(messages, tokenize=False)

In [8]:
mistral_chat_test = mistral_tokenizer.apply_chat_template(messages, tokenize=True)

In [12]:
print(mistral_chat)
print("-" * 50)
print(smol_chat)
print("-" * 50)
print(mistral_chat_test)

<s> [INST] You are a helpful assistant.

Hello! [/INST]
--------------------------------------------------
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>

--------------------------------------------------
{'input_ids': [1, 733, 16289, 28793, 995, 460, 264, 10865, 13892, 28723, 13, 13, 16230, 28808, 733, 28748, 16289, 28793], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [15]:
mistral_chat_test

{'input_ids': [1, 733, 16289, 28793, 995, 460, 264, 10865, 13892, 28723, 13, 13, 16230, 28808, 733, 28748, 16289, 28793], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

### 1.2. Advanced Features


Chat templates can handle more complex scenarios beyond just conversational interactions, including:

1. **Tool Use**: When models need to interact with external tools or APIs

2. **Multimodal Inputs**: For handling images, audio, or other media types

3. **Function Calling**: For structured function execution

4. **Multi-turn Context**: For maintaining conversation history

For multimodal conversations, chat templates can include image references or base64-encoded images:



In [16]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful vision assistant that can analyze images.",
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What's in this image?"},
            {"type": "image", "image_url": "https://example.com/image.jpg"},
        ],
    },
]

Here’s an example of a chat template with tool use:



In [17]:
messages = [
    {
        "role": "system",
        "content": "You are an AI assistant that can use tools. Available tools: calculator, weather_api",
    },
    {"role": "user", "content": "What's 123 * 456 and is it raining in Paris?"},
    {
        "role": "assistant",
        "content": "Let me help you with that.",
        "tool_calls": [
            {
                "tool": "calculator",
                "parameters": {"operation": "multiply", "x": 123, "y": 456},
            },
            {"tool": "weather_api", "parameters": {"city": "Paris", "country": "France"}},
        ],
    },
    {"role": "tool", "tool_name": "calculator", "content": "56088"},
    {
        "role": "tool",
        "tool_name": "weather_api",
        "content": "{'condition': 'rain', 'temperature': 15}",
    },
]

## 2. Best Practices

### General Guidelines


When working with chat templates, follow these key practices:

1. **Consistent Formatting**: Always use the same template format throughout your application

2. **Clear Role Definition**: Clearly specify roles (system, user, assistant, tool) for each message

3. **Context Management**: Be mindful of token limits when maintaining conversation history

4. **Error Handling**: Include proper error handling for tool calls and multimodal inputs
Validation: Validate message structure before sending to the model

Common pitfalls to avoid:

* Mixing different template formats in the same application

* Exceeding token limits with long conversation histories

* Not properly escaping special characters in messages

* Forgetting to validate input message structure

* Ignoring model-specific template requirements